# Day 1 — The Harry Potter Character Network
**YTS+ DSEB 2026 · Project A**

---

Someone read all seven Harry Potter books and recorded every time two characters appeared within 14 words of each other. That record is your data.

Today you will load it, understand what it contains, and compute your first set of statistics.

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os

os.makedirs('images', exist_ok=True)
DATA = 'data/'
BOOKS = ['Stone', 'Secrets', 'Azkaban', 'Fire', 'Phoenix', 'Prince', 'Hallows']

print('Ready.')

---
## Part 1 — What is in the data?

Load the Book 3 edge file. Each row is a pair of characters.

In [ ]:
df3 = pd.read_csv(DATA + 'hp_book3_edges.csv')

print(f'Shape: {df3.shape}')
print(f'Columns: {df3.columns.tolist()}')
print()
df3.head(10)

**Q1 — What does one row mean?** In plain English, what does the `weight` column measure? Is a high weight the same as a close friendship?

*Write your answer here:*

In [ ]:
print('Weight distribution:')
print(df3['weight'].describe().round(1))
print()
print(f'Pairs appearing together only once: {(df3["weight"] == 1).sum()}')
print(f'Pairs appearing 10+ times:          {(df3["weight"] >= 10).sum()}')
print(f'Pairs appearing 50+ times:          {(df3["weight"] >= 50).sum()}')

In [ ]:
print('Top 15 most frequent pairs in Book 3:')
print(df3.sort_values('weight', ascending=False).head(15).to_string(index=False))

**Q2 — Look at the top 15 pairs.** Crabbe and Goyle appear together 95+ times. Does Rowling treat them as two characters or one? What other pairs here could you argue are really a single unit?

*Write your answer here:*

---
## Part 2 — How big is the network?

Count unique characters and pairs across all 7 books.

In [ ]:
print(f'{"Book":<6} {"Title":<10} {"Pairs":>8} {"Characters":>12}')
print('-' * 40)

for i in range(1, 8):
    df = pd.read_csv(DATA + f'hp_book{i}_edges.csv')
    chars = set(df['source']) | set(df['target'])
    print(f'{i:<6} {BOOKS[i-1]:<10} {len(df):>8} {len(chars):>12}')

**Q3 — The network does not grow steadily across 7 books.** It shrinks, then jumps, then stabilises. Pick one inflection point (a book where the number changes sharply) and explain it using your knowledge of the plot.

*Write your answer here:*

---
## Part 3 — Who appears the most?

Build the Book 3 network and compute weighted degree — the total number of co-appearances for each character across all their partners.

In [ ]:
G3 = nx.from_pandas_edgelist(df3, 'source', 'target', edge_attr='weight')

print(f'Characters (nodes): {G3.number_of_nodes()}')
print(f'Pairs (edges):      {G3.number_of_edges()}')

In [ ]:
degree = sorted(G3.degree(weight='weight'), key=lambda x: x[1], reverse=True)

print('Top 20 characters by weighted degree (Book 3):')
for i, (name, d) in enumerate(degree[:20]):
    print(f'  {i+1:>2}. {name:<30} {d}')

In [ ]:
names  = [n for n, _ in degree[:15]]
values = [v for _, v in degree[:15]]

plt.figure(figsize=(9, 6))
plt.barh(names[::-1], values[::-1], color='steelblue')
plt.xlabel('Weighted degree (total co-appearances)')
plt.title('Book 3 — Top 15 characters by weighted degree')
plt.tight_layout()
plt.savefig('images/day1_degree.png', dpi=150)
plt.show()

**Q4 — Before you looked, who did you expect to be in the top 5?** Now that you have the list: which rankings match your intuition, and which don't? What does weighted degree measure that your intuition about "importance" does not?

*Write your answer here:*

---
## Part 4 — Remove Harry

Harry dominates the degree list. What happens when you remove him entirely?

In [ ]:
G3_no_harry = G3.copy()
G3_no_harry.remove_node('Harry Potter')

degree_nh = sorted(G3_no_harry.degree(weight='weight'), key=lambda x: x[1], reverse=True)

print('Top 10 characters by weighted degree — Harry removed:')
for i, (name, d) in enumerate(degree_nh[:10]):
    print(f'  {i+1:>2}. {name:<30} {d}')

**Q5 — Does the ranking change when Harry is removed, or does the same person come second?** Who leads the network without Harry — and does that match how you read Book 3?

*Write your answer here:*

---
## What you did today

- Loaded a co-appearance network from raw edge data
- Computed basic statistics: weight distribution, character counts per book, weighted degree
- Saw that the network contracts and expands in ways that trace Rowling's plot decisions
- Removed the dominant node and asked what remains

**Next session:** community detection — the algorithm will partition the network into groups using only edge weights, with no knowledge of the plot.